In [9]:
from typing_extensions import override
from rich.console import Console
from dotenv import load_dotenv
from groq import Groq
import json
load_dotenv(override=True)


True

In [10]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [38]:

client = Groq()

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages
)
print(response.choices[0].message.content)


[**Step 1: Estimate the time it takes for the first train to travel some reasonable distance**]
Since we don't know the exact distance between Boston and New York, we'll estimate the distance as 200 miles. This is a rough estimate of a typical distance from Boston to New York (approximately around 215 miles).

[**Step 2: Calculate the time it takes for the first train to travel the estimated distance**]
Time = Distance / Speed
Time = 200 miles / 60 mph
Time = 3.33 hours
Let's convert this to hours and minutes: 3 hours and 20 minutes.

[**Step 3: Calculate the time difference between the two trains departure times**]
New York train departs 1 hour after Boston train (3:00 pm - 2:00 pm).
Let's call the time at which the trains meet 't'. The time elapsed since the New York train departed is 't' hours. 

[**Step 4: Calculate the time elapsed since the Boston train departed**]
Since the total time elapsed since both trains departed is 't', and the New York train departed 1 hour later than th

In [12]:
from groq import Groq

client = Groq()

models = client.models.list()

for m in models.data:
    print(m.id)


meta-llama/llama-4-maverick-17b-128e-instruct
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-safeguard-20b
moonshotai/kimi-k2-instruct-0905
canopylabs/orpheus-arabic-saudi
groq/compound-mini
allam-2-7b
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
llama-3.1-8b-instant
openai/gpt-oss-120b
qwen/qwen3-32b
groq/compound
openai/gpt-oss-20b
llama-3.3-70b-versatile
meta-llama/llama-guard-4-12b
meta-llama/llama-4-scout-17b-16e-instruct
whisper-large-v3
moonshotai/kimi-k2-instruct


In [13]:
todo = ["Buy milk", "Study Python", "Workout"]
completed = [True, False, True]


In [14]:
def get_todo_report() -> str:
    result = ""

    for index, task in enumerate(todo):
        if completed[index]:
            result += f"Todo #{index+1}: {task} (done)\n"
        else:
            result += f"Todo #{index+1}: {task}\n"

    print(result)
    return result


In [15]:
get_todo_report()

Todo #1: Buy milk (done)
Todo #2: Study Python
Todo #3: Workout (done)



'Todo #1: Buy milk (done)\nTodo #2: Study Python\nTodo #3: Workout (done)\n'

In [16]:
def create_todos(descriptions:list[str])-> str:
    todos.extend(descriptions)
    completed.extend([False]* len(descriptions))
    return get_todo_report()
    


In [17]:
def mark_complete(index:int, completion_notes:str)->str:
    if 1<=index<= len(todos):
        completed[index-1]=True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [18]:
todos,completed=[],[]
create_todos(["Buy Groceries","Finish extra lab","Eat banana"])

Todo #1: Buy milk
Todo #2: Study Python
Todo #3: Workout



'Todo #1: Buy milk\nTodo #2: Study Python\nTodo #3: Workout\n'

In [19]:
mark_complete(1,"bought")

bought

Todo #1: Buy milk (done)
Todo #2: Study Python
Todo #3: Workout



'Todo #1: Buy milk (done)\nTodo #2: Study Python\nTodo #3: Workout\n'

In [20]:
create_todos_json = {
    "type": "function",
    "function": {
        "name": "create_todos",
        "description": "Add new todos from a list of descriptions and return the full list",
        "parameters": {
            "type": "object",
            "properties": {
                "descriptions": {
                    "type": "array",
                    "items": {
                        "type": "string"
                    },
                    "description": "List of todo descriptions"
                }
            },
            "required": ["descriptions"],
            "additionalProperties": False
        }
    }
}


In [31]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "create_todos",
            "description": "Create multiple todos",
            "parameters": {
                "type": "object",
                "properties": {
                    "descriptions": {
                        "type": "array",
                        "items": {"type": "string"}
                    }
                },
                "required": ["descriptions"]
            }
        }
    }
]


In [32]:
import json

def handle_tool_calls(tool_calls):
    results = []

    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}

        results.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result)
        })

    return results


In [33]:
print(tools)
for t in tools:
    print(type(t))


[{'type': 'function', 'function': {'name': 'create_todos', 'description': 'Create multiple todos', 'parameters': {'type': 'object', 'properties': {'descriptions': {'type': 'array', 'items': {'type': 'string'}}}, 'required': ['descriptions']}}}]
<class 'dict'>


In [34]:
def loop(messages):
    done = False
    while not done:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            tools=tools
        )

        choice = response.choices[0]

        if choice.finish_reason == "tool_calls":
            msg = choice.message
            calls = msg.tool_calls

            results = []
            for call in calls:
                fn = globals()[call.function.name]
                args = json.loads(call.function.arguments)
                result = fn(**args)
                results.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": json.dumps(result)
                })

            messages.append(msg)
            messages.extend(results)
        else:
            done = True
            print(choice.message.content)


In [35]:
from groq import Groq
import json

client = Groq()


In [39]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out the steps.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""

user_message = """
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message}
]



In [50]:
todos,completed=[],[]
loop(messages)

The trains will meet at 5:20 pm.
